# Corrected Data Merge - Patient Demographics + Visit Data

This notebook properly merges:
1. Patient-level demographics (one record per patient from TRANS event)
2. Visit-level data (Age at Visit + DaTScan)

Key fixes:
- Sex and birth date are constant per patient (no duplicates)
- Age, birth date, and visit date are reconciled
- Each patient has exactly one demographic record merged with all their visits

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load data
data_dir = Path('data/csvData')
df_age = pd.read_csv(data_dir / 'Age_at_visit_14Oct2025.csv')
df_demographics = pd.read_csv(data_dir / 'Demographics_14Oct2025.csv')
df_datscan = pd.read_csv(data_dir / 'DaTScan_SBR_Analysis_17Dec2024.csv')

print(f"Loaded:")
print(f"  Age at Visit: {df_age.shape}")
print(f"  Demographics: {df_demographics.shape}")
print(f"  DaTScan: {df_datscan.shape}")

In [ ]:
# Helper function to parse MM/YYYY dates
def parse_date(date_str):
    """Parse MM/YYYY format to datetime"""
    if pd.isna(date_str) or date_str == '':
        return None
    try:
        return pd.to_datetime(date_str, format='%m/%Y')
    except:
        return None

# Step 1: Extract patient-level demographics (one per patient)
# Priority: TRANS event (baseline screening), then first valid record
print("\n" + "="*80)
print("STEP 1: Extract Patient-Level Demographics")
print("="*80)

# Get TRANS records (baseline demographics)
df_demo_trans = df_demographics[df_demographics['EVENT_ID'] == 'TRANS'].copy()
print(f"Patients with TRANS event: {len(df_demo_trans)}")

# For patients missing TRANS, use first record with valid SEX and BIRTHDT
patients_with_trans = set(df_demo_trans['PATNO'].unique())
all_patients = set(df_demographics['PATNO'].unique())
missing_trans = all_patients - patients_with_trans

if len(missing_trans) > 0:
    print(f"Patients missing TRANS event: {len(missing_trans)}")
    df_demo_backup = df_demographics[df_demographics['PATNO'].isin(missing_trans)].copy()
    # Keep only records with valid SEX and BIRTHDT
    df_demo_backup = df_demo_backup.dropna(subset=['SEX', 'BIRTHDT'])
    # Sort by PATNO and EVENT_ID, take first
    df_demo_backup = df_demo_backup.sort_values(['PATNO', 'EVENT_ID']).groupby('PATNO').first().reset_index()
    df_demo_patient = pd.concat([df_demo_trans, df_demo_backup], ignore_index=True)
else:
    df_demo_patient = df_demo_trans.copy()

print(f"Total patient-level demographic records: {len(df_demo_patient)}")
print(f"Unique patients: {df_demo_patient['PATNO'].nunique()}")

# Validate: each patient should have exactly one record
dup_check = df_demo_patient.groupby('PATNO').size()
if (dup_check > 1).any():
    print(f"WARNING: {(dup_check > 1).sum()} patients have multiple demographic records!")
else:
    print("✓ Each patient has exactly one demographic record")

In [ ]:
# Step 2: Validate demographic consistency
print("\n" + "="*80)
print("STEP 2: Validate Demographic Consistency")
print("="*80)

# Check SEX consistency
sex_nunique = df_demo_patient.groupby('PATNO')['SEX'].nunique()
print(f"Patients with unique SEX value: {(sex_nunique == 1).sum()}")
if (sex_nunique > 1).any():
    print(f"  WARNING: {(sex_nunique > 1).sum()} patients have multiple SEX values")

# Check BIRTHDT consistency
birthdt_nunique = df_demo_patient.groupby('PATNO')['BIRTHDT'].nunique()
print(f"Patients with unique BIRTHDT value: {(birthdt_nunique == 1).sum()}")
if (birthdt_nunique > 1).any():
    print(f"  WARNING: {(birthdt_nunique > 1).sum()} patients have multiple BIRTHDT values")

# Show sample
print(f"\nSample patient demographics:")
print(df_demo_patient[['PATNO', 'BIRTHDT', 'SEX', 'INFODT']].head(10))

In [ ]:
# Step 3: Merge Age + DaTScan (visit-level data)
print("\n" + "="*80)
print("STEP 3: Merge Visit-Level Data (Age + DaTScan)")
print("="*80)

# Merge on PATNO + EVENT_ID
df_visit = pd.merge(
    df_age,
    df_datscan,
    on=['PATNO', 'EVENT_ID'],
    how='outer'
)

print(f"Visit-level records (Age outer join DaTScan): {len(df_visit)}")
print(f"Unique patients: {df_visit['PATNO'].nunique()}")
print(f"Unique events: {df_visit['EVENT_ID'].nunique()}")

In [ ]:
# Step 4: Merge visit data with patient demographics
print("\n" + "="*80)
print("STEP 4: Merge Visit Data with Patient Demographics")
print("="*80)

# Select demographic columns to merge
demo_cols = ['PATNO', 'BIRTHDT', 'SEX', 'REC_ID', 'PAG_NAME', 'INFODT', 
             'AFICBERB', 'ASHKJEW', 'BASQUE', 'CHLDBEAR', 'HOWLIVE', 
             'GAYLES', 'HETERO', 'BISEXUAL', 'PANSEXUAL', 'ASEXUAL', 
             'OTHSEXUALITY', 'HANDED', 'HISPLAT', 'RAASIAN', 'RABLACK', 
             'RAHAWOPI', 'RAINDALS', 'RANOS', 'RAWHITE', 'RAUNKNOWN', 
             'ORIG_ENTRY', 'LAST_UPDATE']

df_demo_patient_slim = df_demo_patient[demo_cols].copy()

# Left join on PATNO only (each patient gets their demographics merged with all visits)
df_merged = pd.merge(
    df_visit,
    df_demo_patient_slim,
    on='PATNO',
    how='left'
)

print(f"Final merged dataset: {df_merged.shape}")
print(f"Unique patients: {df_merged['PATNO'].nunique()}")
print(f"Unique events: {df_merged['EVENT_ID'].nunique()}")

# Validate: each patient should have constant SEX and BIRTHDT
sex_check = df_merged.groupby('PATNO')['SEX'].nunique()
birthdt_check = df_merged.groupby('PATNO')['BIRTHDT'].nunique()

print(f"\nValidation after merge:")
print(f"  Patients with constant SEX: {(sex_check <= 1).sum()}")
print(f"  Patients with constant BIRTHDT: {(birthdt_check <= 1).sum()}")

if (sex_check > 1).any():
    print(f"  ✗ ERROR: {(sex_check > 1).sum()} patients have multiple SEX values!")
else:
    print(f"  ✓ All patients have constant SEX")

if (birthdt_check > 1).any():
    print(f"  ✗ ERROR: {(birthdt_check > 1).sum()} patients have multiple BIRTHDT values!")
else:
    print(f"  ✓ All patients have constant BIRTHDT")

In [ ]:
# Step 5: Reconcile age, birth date, and visit date
print("\n" + "="*80)
print("STEP 5: Reconcile Age, Birth Date, and Visit Date")
print("="*80)

# Parse dates
df_merged['BIRTHDT_parsed'] = df_merged['BIRTHDT'].apply(parse_date)
df_merged['INFODT_parsed'] = df_merged['INFODT'].apply(parse_date)

# Calculate age from birth date and visit date
df_merged['AGE_from_dates'] = (df_merged['INFODT_parsed'] - df_merged['BIRTHDT_parsed']).dt.days / 365.25

# Check consistency between AGE_AT_VISIT and calculated age
has_both = df_merged['AGE_AT_VISIT'].notna() & df_merged['AGE_from_dates'].notna()
df_merged['age_discrepancy'] = abs(df_merged['AGE_AT_VISIT'] - df_merged['AGE_from_dates'])

print(f"Records with AGE_AT_VISIT: {df_merged['AGE_AT_VISIT'].notna().sum()}")
print(f"Records with calculated age from dates: {df_merged['AGE_from_dates'].notna().sum()}")
print(f"Records with both: {has_both.sum()}")

if has_both.sum() > 0:
    print(f"\nAge consistency check (records with both values):")
    print(f"  Mean discrepancy: {df_merged.loc[has_both, 'age_discrepancy'].mean():.3f} years")
    print(f"  Max discrepancy: {df_merged.loc[has_both, 'age_discrepancy'].max():.3f} years")
    print(f"  Records with discrepancy < 0.5 years: {(df_merged.loc[has_both, 'age_discrepancy'] < 0.5).sum()}")
    print(f"  Records with discrepancy >= 0.5 years: {(df_merged.loc[has_both, 'age_discrepancy'] >= 0.5).sum()}")
    
    # Show examples of large discrepancies
    large_disc = df_merged[has_both & (df_merged['age_discrepancy'] >= 0.5)].sort_values('age_discrepancy', ascending=False)
    if len(large_disc) > 0:
        print(f"\n  Examples of large discrepancies (>= 0.5 years):")
        for idx, row in large_disc.head(5).iterrows():
            print(f"    PATNO {row['PATNO']}, EVENT {row['EVENT_ID']}: AGE_AT_VISIT={row['AGE_AT_VISIT']:.1f}, calculated={row['AGE_from_dates']:.1f}, diff={row['age_discrepancy']:.2f}")

In [ ]:
# Step 6: Create reconciled age column
print("\n" + "="*80)
print("STEP 6: Create Reconciled Age Column")
print("="*80)

# Strategy:
# 1. Use AGE_AT_VISIT if available
# 2. If missing, use calculated age from dates
# 3. Flag large discrepancies for review

df_merged['AGE_AT_VISIT_reconciled'] = df_merged['AGE_AT_VISIT'].copy()

# Fill missing AGE_AT_VISIT with calculated age
missing_age = df_merged['AGE_AT_VISIT'].isna() & df_merged['AGE_from_dates'].notna()
df_merged.loc[missing_age, 'AGE_AT_VISIT_reconciled'] = df_merged.loc[missing_age, 'AGE_from_dates']

print(f"Age reconciliation:")
print(f"  Records with AGE_AT_VISIT (original): {df_merged['AGE_AT_VISIT'].notna().sum()}")
print(f"  Records filled from calculated age: {missing_age.sum()}")
print(f"  Total records with reconciled age: {df_merged['AGE_AT_VISIT_reconciled'].notna().sum()}")

# Flag inconsistent records
df_merged['age_inconsistent'] = has_both & (df_merged['age_discrepancy'] >= 0.5)
print(f"  Records flagged as inconsistent (discrepancy >= 0.5 years): {df_merged['age_inconsistent'].sum()}")

In [ ]:
# Step 7: Prepare final output
print("\n" + "="*80)
print("STEP 7: Prepare Final Output")
print("="*80)

# Select columns for output (drop intermediate columns)
output_cols = [
    'PATNO', 'EVENT_ID', 'AGE_AT_VISIT_reconciled',
    'BIRTHDT', 'SEX', 'INFODT',
    'REC_ID', 'PAG_NAME',
    'AFICBERB', 'ASHKJEW', 'BASQUE', 'CHLDBEAR', 'HOWLIVE',
    'GAYLES', 'HETERO', 'BISEXUAL', 'PANSEXUAL', 'ASEXUAL',
    'OTHSEXUALITY', 'HANDED', 'HISPLAT', 'RAASIAN', 'RABLACK',
    'RAHAWOPI', 'RAINDALS', 'RANOS', 'RAWHITE', 'RAUNKNOWN',
    'ORIG_ENTRY', 'LAST_UPDATE',
    'PROTOCOL', 'DATSCAN_LIGAND', 'DATSCAN_DATE',
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT',
    'DATSCAN_ANALYZED', 'DATSCAN_NOT_ANALYZED_REASON', 'DATSCAN_OTHER_SPECIFY'
]

df_output = df_merged[output_cols].copy()

# Rename reconciled age column
df_output = df_output.rename(columns={'AGE_AT_VISIT_reconciled': 'AGE_AT_VISIT'})

print(f"Output dataset shape: {df_output.shape}")
print(f"Columns: {df_output.columns.tolist()}")

# Show sample
print(f"\nSample output (first patient):")
sample_patno = df_output['PATNO'].iloc[0]
sample = df_output[df_output['PATNO'] == sample_patno][['PATNO', 'EVENT_ID', 'AGE_AT_VISIT', 'BIRTHDT', 'SEX', 'INFODT']].head(10)
print(sample)

In [ ]:
# Step 8: Save output
print("\n" + "="*80)
print("STEP 8: Save Output")
print("="*80)

output_path = Path('output/merged_data_corrected.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_output.to_csv(output_path, index=False)

print(f"✓ Saved to: {output_path}")
print(f"  Shape: {df_output.shape}")
print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# Step 9: Final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nDataset Statistics:")
print(f"  Total records: {len(df_output)}")
print(f"  Unique patients: {df_output['PATNO'].nunique()}")
print(f"  Unique events: {df_output['EVENT_ID'].nunique()}")
print(f"  Avg records per patient: {len(df_output) / df_output['PATNO'].nunique():.2f}")

print(f"\nData Completeness:")
print(f"  With age: {df_output['AGE_AT_VISIT'].notna().sum()} ({df_output['AGE_AT_VISIT'].notna().sum()/len(df_output)*100:.1f}%)")
print(f"  With demographics (SEX): {df_output['SEX'].notna().sum()} ({df_output['SEX'].notna().sum()/len(df_output)*100:.1f}%)")
print(f"  With DaTScan: {df_output['DATSCAN_CAUDATE_R'].notna().sum()} ({df_output['DATSCAN_CAUDATE_R'].notna().sum()/len(df_output)*100:.1f}%)")

print(f"\nDemographic Consistency:")
sex_const = df_output.groupby('PATNO')['SEX'].nunique().eq(1).sum()
birthdt_const = df_output.groupby('PATNO')['BIRTHDT'].nunique().eq(1).sum()
print(f"  Patients with constant SEX: {sex_const}/{df_output['PATNO'].nunique()}")
print(f"  Patients with constant BIRTHDT: {birthdt_const}/{df_output['PATNO'].nunique()}")

if sex_const == df_output['PATNO'].nunique() and birthdt_const == df_output['PATNO'].nunique():
    print(f"\n✓ SUCCESS: All demographic consistency checks passed!")
else:
    print(f"\n✗ ERROR: Demographic consistency issues detected!")